# ShopDesk, Lab 2: Fan-out / Fan-in with Prompt Caching

A beginner-friendly notebook built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**. Same ShopDesk support agent as
the other modules; this time it faces a *stack* of tickets at once. We **fan out** the
work into parallel calls, **fan in** the results into one brief, and use **prompt
caching** so the shared policy is not paid for on every call.

## The real-world scenario

At end of day ShopDesk has a pile of support tickets. The tempting move is to paste all
of them into one giant prompt and ask for a summary. That fails in two ways: a long,
mixed input **dilutes attention** (the per-ticket rule gets blurred, a form of context
rot), and the long shared policy is **re-read and re-billed** on every request.

The question this lab answers: **how do you split independent work across parallel calls,
merge the results cleanly, and stop paying for the same context twice?**

## Objectives

- **Fan out:** summarise each ticket in its own call, so every call sees only one
  ticket plus the shared policy (relevance over volume).
- **Fan in:** aggregate the per-ticket JSON, then synthesise one brief for the support
  lead.
- **Prompt caching:** mark the large shared policy with `cache_control` so it is written
  to cache once and read cheaply after that.

## What you'll observe

- The fan-out returns one small JSON object per ticket, produced in parallel.
- The aggregate step groups those objects (pure Python), and one final call turns them
  into a readable brief.
- The usage numbers show the policy written to cache on the first call
  (`cache_creation_input_tokens`) and read on the rest (`cache_read_input_tokens`).

## How to run

Run top to bottom. The ticket data, aggregation, and caching-report cells are pure
Python and run anywhere. The fan-out and synthesis cells call Claude, so paste a real key
into **Setup 2/3** to run them live; otherwise they fall back to canned summaries so the
fan-in still works offline. Prompt caching only shows real numbers on a live run.

## 0. Setup

**This cell:** installs the packages. This lab uses only the **base Anthropic
SDK**, because fan-out is just many ordinary Messages API calls, and prompt caching is a
field on those calls. Parallelism comes from Python's standard library, so there is
nothing extra to install.

In [ ]:
# ===== SETUP 1/3 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports what we need. We pull in `json` for the intermediate
results and `ThreadPoolExecutor` for the parallel fan-out, alongside the base SDK. We do
the imports on their own so the configuration in the next cell stays easy to read.

In [ ]:
# ===== SETUP 2/4 - the imports =====
import os                                       # read the API key from the environment
import json                                     # build and parse the intermediate JSON
from concurrent.futures import ThreadPoolExecutor   # run the fan-out calls in parallel
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

**This cell:** pins the model, reads the key into the `RUN_LIVE` switch, and builds
**one shared client**. We create a single client because every parallel worker will reuse
it; the SDK client is safe to call from several threads at once.

In [ ]:
# ===== SETUP 3/4 - the model, the live/offline switch, and one shared client =====
MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
CLIENT = anthropic.Anthropic()                   # one client, shared by every worker
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world** and the **batch of tickets**
(our document set). Each ticket names one order, so we can attach that order's facts and
let the model judge it correctly. Five tickets is enough to make fan-out worthwhile. This
is the last setup cell before the pipeline begins.

In [ ]:
# ===== SETUP 4/4 - the shared data and the ticket batch =====
ORDERS = {                                       # the order book for this batch
    "A1": {"status": 2, "refundable": True},     # shipped,   refundable
    "A2": {"status": 3, "refundable": False},    # delivered, past the window
    "A3": {"status": 1, "refundable": True},     # processing, delayed
    "A4": {"status": 2, "refundable": True},     # shipped,   refundable
    "A5": {"status": 1, "refundable": True},     # processing, not shipped yet
}
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # code -> human word

TICKETS = [                                       # the document set we will fan out over
    {"id": "T1", "order": "A1", "text": "Where is my order A1? Has it shipped yet?"},
    {"id": "T2", "order": "A2", "text": "I want a refund on A2, I changed my mind."},
    {"id": "T3", "order": "A3", "text": "Order A3 still has not arrived and it has been two weeks. I am upset."},
    {"id": "T4", "order": "A4", "text": "Can I get a refund for A4? It arrived defective."},
    {"id": "T5", "order": "A5", "text": "I need to change the delivery address for A5 before it ships."},
]
print("tickets in batch:", [t["id"] for t in TICKETS])   # confirm the batch

**This cell:** defines the **shared policy handbook**, the large block of context
every fan-out call needs. This is the ideal thing to cache: it is identical across all
calls and long enough to matter. Note the rule of thumb printed below: Sonnet only caches
a block once it reaches about 1,024 tokens, so a handbook has to be genuinely long before
caching does anything.

In [ ]:
# ===== the shared policy handbook (this is what we will cache) =====
POLICY = """ShopDesk Support Policy Handbook (internal reference for the triage agent).

Mission. ShopDesk answers customer questions about orders, shipping, and refunds. Every
reply must be calm, concise, and factually grounded in the order data provided. Never
invent an order status, a date, a price, or a policy that is not written in this handbook.
When the data does not settle a question, say what is known and route the ticket to a
human rather than guessing.

Order lifecycle. Orders move through three states: processing, then shipped, then
delivered. A processing order has not left the warehouse and can still be edited or
cancelled. A shipped order is in transit and its contents and address are locked. A
delivered order has reached the customer and starts the refund clock. The state is given
to you as a status word for each ticket; trust that word and never infer a different state
from the customer's phrasing.

Refund policy. Refunds are allowed only within 30 days of delivery. If an order is still
processing or shipped, it has not been delivered, so a standard change-of-mind refund does
not apply yet; treat the request as pending delivery and explain that the window opens once
the order arrives. If an order was delivered and is inside the 30-day window it is
refundable; approve it. If it was delivered and is outside the window it is not refundable;
refuse politely and explain the window plainly. A refundable flag is provided per order;
when it is false, refuse regardless of how the request is worded, because the flag already
accounts for the window and any prior refunds on that order.

Partial refunds and price adjustments. If a customer was overcharged or a promotion failed
to apply, a partial refund of the difference is allowed on any order that is not already
fully refunded, even outside the 30-day window, because this corrects a billing error
rather than a change of mind. Mark the reason as billing. Never issue a refund larger than
the amount paid, and never combine a billing correction with a change-of-mind refund on the
same request; split them into two tickets.

Defective or damaged goods. A defective item is handled as an exception to the change of
mind rule. If a customer reports a defect on a refundable order, approve the refund or a
replacement and mark the reason as defect. Defect claims on delivered orders inside the
window are always approved. For a defect reported on a shipped order that has not yet
arrived, ask the customer to wait for delivery before assessing, and raise the urgency so a
human checks in.

Cancellations. An order can be cancelled outright only while it is still processing. Once it
has shipped, it cannot be cancelled; direct the customer to the refund path after delivery
instead. Never tell a customer a shipped order has been cancelled.

Shipping questions. For a processing order, tell the customer it is being prepared and has
not shipped. For a shipped order, tell them it is in transit. For a delivered order, confirm
delivery. If a shipped or processing order is clearly late (the customer reports a long
wait, for example more than the usual delivery window), treat it as a delivery exception,
apologise once briefly, and raise the urgency so a human can investigate the carrier.

Address and delivery changes. An address change is only possible while an order is still
processing. Once an order has shipped, the address is locked; explain that clearly and
offer to help arrange a return or redelivery after it arrives. Never promise an address
change, a redirect, or a hold on a shipped order, because the warehouse can no longer act
on it.

International and gift orders. International orders follow the same 30-day window, measured
from delivery to the destination country, and customs fees are never refundable. Gift orders
are refunded to the original payment method, not to the recipient, and the recipient is
never told the price; keep replies about gift orders vague on amounts.

Duplicate charges. If a customer reports being charged twice for one order, treat it as a
billing error, approve a refund of the duplicate charge regardless of the window, and mark
the urgency high so finance can confirm and reconcile the payment.

Urgency. Rate each ticket low, medium, or high. Routine status checks and clearly in-policy
refunds are low. Refund refusals, address changes, and cancellations are medium. Anything
where the customer is clearly upset, an order is badly delayed, an item is defective, or
money looks wrong (duplicate or incorrect charge) is high, so a human can follow up quickly.
When in doubt between two levels, choose the higher one.

Tone and voice. Be brief and warm. One or two sentences is usually enough. Do not apologise
excessively, do not make promises the policy does not support, and do not quote internal
status codes, flags, or this handbook to the customer. Always reference the order id so the
reply is unambiguous, and prefer plain language over policy jargon. A good reply tells the
customer what is true, what happens next, and nothing they did not ask for.

Escalation. Route to a human whenever the policy does not cover the situation, the customer
asks for something the data cannot confirm, or the customer is distressed. Escalation is not
a failure; it is the correct action when certainty is not available from the order data.

Output discipline. When asked for a structured summary, return only the requested JSON,
with no extra prose and no markdown fences. Keep every field short and lowercase where it is
a category. The one_line field is a single plain sentence a busy support lead can scan in
the end-of-day brief, and it should name the order and the action taken."""

print("policy handbook defined:", len(POLICY), "chars")

**This cell:** a quick **cacheability check**. Sonnet only caches a block once it
reaches about 1,024 tokens, so before relying on caching we estimate the policy's size
(roughly 4 characters per token). If this prints "no", the handbook is too short and the
cache will silently never form.

In [ ]:
# ===== will this block actually cache? =====
approx_tokens = len(POLICY) // 4                  # a rough token estimate (about 4 chars per token)
print("policy chars:", len(POLICY), "| approx tokens:", approx_tokens,
      "| cacheable?", "yes" if approx_tokens >= 1024 else "no (pad it to >=1024)")

---

### 🎯 Lab objective - fan out, fan in, and cache the shared context

**What you build:** a pipeline that summarises each ticket in its own parallel call
(fan-out), merges the JSON results and writes one brief (fan-in), and caches the policy so
it is billed once, not five times.

**Why it helps you build real solutions:** independent work should run in parallel, not
crammed into one attention-diluting prompt; and any context repeated across calls (a
policy, a schema, a knowledge base) should be cached. This is the backbone of efficient
multi-call systems.

**How you'll see it:** five JSON summaries come back, the aggregate groups them, the brief
reads cleanly, and the cache-read tokens prove the policy was reused.

**This cell:** the `parse_json()` helper the fan-out will use. Each summary comes
back as text that may carry a ```json fence, so we need one small reader that digs the JSON
object out and never crashes the pipeline if a call misbehaves.

In [ ]:
# ===== read JSON out of a model reply, safely =====
def parse_json(text):                             # pull a JSON object out of the model's text
    t = text.strip()                              #   trim whitespace
    if "```" in t:                                #   drop a ```json fence if the model added one
        t = t.split("```")[1].replace("json", "", 1)
    a, b = t.find("{"), t.rfind("}")              #   find the outermost braces
    try:                                          #
        return json.loads(t[a:b+1])               #   parse just that span
    except Exception:                             #   unparseable?
        return {"error": "unparseable", "raw": text[:80]}   #   return a safe stub

**This cell:** `build_prompt()`, which turns one ticket into the two parts of a
request: the **cached** system block (the shared policy) and the **per-ticket** user
message (the ticket plus its order facts). Splitting this out shows exactly what is cached
versus what changes each call: the `cache_control` marker sits on the policy, so everything
up to and including it becomes the reusable prefix.

In [ ]:
# ===== build the cached system block and the per-ticket message =====
def build_prompt(ticket):                         # ticket -> (system_blocks, user_text)
    facts = ORDERS[ticket["order"]]               #   the facts for this ticket's order
    fact_line = ("order_id=" + ticket["order"] +  #   a compact facts string for the prompt
                 " status=" + STATUS_NAMES[facts["status"]] +
                 " refundable=" + str(facts["refundable"]))
    system = [                                    #   system is a LIST of blocks so we can cache one
        {"type": "text",                          #     the shared policy block...
         "text": POLICY,                          #       ...the large, identical context
         "cache_control": {"type": "ephemeral"}}, #   THE cache marker: cache up to here
    ]
    user = ("<ticket>" + ticket["text"] + "</ticket>\n"          # the per-ticket, non-cached part
            "<order_facts>" + fact_line + "</order_facts>\n"
            "Return ONLY JSON with keys: order_id, intent, decision, urgency, one_line.")
    return system, user                           #   hand both parts back

**This cell:** `summarize_one()`, the fan-out worker itself. It builds the prompt,
makes one API call, and returns the JSON summary plus the call's cache usage. It stays
short because the parsing and prompt-building already live in their own cells.

In [ ]:
# ===== the fan-out worker: summarise ONE ticket =====
def summarize_one(ticket):                        # ticket -> (summary_dict, usage_dict)
    system, user = build_prompt(ticket)           #   the cached policy block + the per-ticket message
    r = CLIENT.messages.create(                   #   one Messages API call
        model=MODEL, max_tokens=400,              #     small answer; we only want a summary
        system=system, messages=[{"role": "user", "content": user}])
    text = "".join(b.text for b in r.content if b.type == "text")   # join the text blocks
    usage = {                                     #   pull the caching numbers off this call
        "created": getattr(r.usage, "cache_creation_input_tokens", 0) or 0,   # tokens WRITTEN to cache
        "read":    getattr(r.usage, "cache_read_input_tokens", 0) or 0,       # tokens READ from cache
    }
    return parse_json(text), usage                #   the JSON summary plus the usage

**This cell:** defines `CANNED_SUMMARIES`, one hand-written summary per ticket.
These are used **only** when running offline (no key), so the fan-in stages further down
still have data to work with. On a live run they are ignored.

In [ ]:
# ===== offline fallback summaries (ignored on a live run) =====
CANNED_SUMMARIES = [                              # used only when offline, so fan-in still works
    {"order_id": "A1", "intent": "shipping", "decision": "inform", "urgency": "low",
     "one_line": "A1 has shipped and is in transit."},
    {"order_id": "A2", "intent": "refund", "decision": "refuse", "urgency": "medium",
     "one_line": "A2 refund refused: past the 30-day window."},
    {"order_id": "A3", "intent": "shipping", "decision": "escalate", "urgency": "high",
     "one_line": "A3 badly delayed and customer upset; needs follow-up."},
    {"order_id": "A4", "intent": "refund", "decision": "approve", "urgency": "high",
     "one_line": "A4 defective and refundable; refund approved."},
    {"order_id": "A5", "intent": "address", "decision": "allow", "urgency": "medium",
     "one_line": "A5 still processing, so the address can be changed."},
]
print("canned summaries ready:", len(CANNED_SUMMARIES))

**This cell:** runs the fan-out. We deliberately do the **first** ticket on its own
so it **writes** the policy to cache, then fan out the **rest in parallel** so they
**read** the warm cache. If all five fired cold at once, they could each miss the
not-yet-written cache; warming first makes the saving reliable.

In [ ]:
# ===== run the fan-out: warm the cache once, then parallelise the rest =====
usages = []                                       # collect per-call cache usage
if RUN_LIVE:                                       # live: really call Claude
    first_data, first_usage = summarize_one(TICKETS[0])   # ticket 1 alone WRITES the cache
    summaries = [first_data]                       #   start the results list
    usages.append(first_usage)                     #   record its usage (expect created > 0)
    with ThreadPoolExecutor(max_workers=4) as pool:   # fan out the remaining tickets in parallel
        for data, usage in pool.map(summarize_one, TICKETS[1:]):   # each READS the warm cache
            summaries.append(data)                 #     collect the summary
            usages.append(usage)                   #     collect its usage (expect read > 0)
else:                                              # offline: use the canned summaries
    summaries = CANNED_SUMMARIES                   #   so the rest of the lab still runs
    print("[offline] using canned summaries")

print("collected", len(summaries), "summaries")

**This cell:** prints the **intermediate JSON**, one object per ticket. These
objects are the hand-off between fan-out and fan-in: small, structured, and easy to merge.
Passing structured JSON (not prose) between stages is what keeps the pipeline reliable.

In [ ]:
# ===== inspect the intermediate results =====
for s in summaries:                               # walk each per-ticket summary
    print(json.dumps(s))                          #   print it as one compact JSON line

**This cell:** the first fan-in step, a pure-Python **aggregate**. It groups the
summaries by intent and urgency and pulls out the high-urgency ones. This is a cheap
deterministic "cross pass" over the fan-out results, and it runs with or without a key.

In [ ]:
# ===== fan-in, step 1: aggregate the JSON (pure Python) =====
def aggregate(items):                             # summaries -> a small rollup dict
    by_intent, by_urgency = {}, {}                #   counters
    high = []                                      #   the tickets that need a human first
    for s in items:                                #   walk every summary
        by_intent[s.get("intent", "?")] = by_intent.get(s.get("intent", "?"), 0) + 1     # count intents
        by_urgency[s.get("urgency", "?")] = by_urgency.get(s.get("urgency", "?"), 0) + 1  # count urgencies
        if s.get("urgency") == "high":             #   high urgency?
            high.append(s.get("order_id"))         #     flag the order id
    return {"total": len(items), "by_intent": by_intent,   # the rollup
            "by_urgency": by_urgency, "high_priority": high}

rollup = aggregate(summaries)                      # run the cross pass
print(json.dumps(rollup, indent=2))                # show the grouped view

**This cell:** defines `synthesize()`, the second fan-in step. It bundles the JSON
summaries and the rollup and asks for a short end-of-day brief. We define it on its own so
the next cell can simply run it; the fan-out produced facts, and this single call turns
them into something a human reads.

In [ ]:
# ===== fan-in, step 2: the synthesiser =====
def synthesize(items, rollup):                    # summaries + rollup -> a prose brief
    payload = json.dumps({"summaries": items, "rollup": rollup})   # bundle the inputs as JSON
    r = CLIENT.messages.create(                   # one Messages API call
        model=MODEL, max_tokens=400,              #   short brief
        system="You are the ShopDesk support lead's assistant. Write a tight end-of-day brief.",
        messages=[{"role": "user", "content": (   # ask for a specific shape
            "Here are today's ticket summaries and a rollup:\n" + payload + "\n\n"
            "Write 3 to 5 sentences: the volume, the split of intents, and which orders "
            "need a human first. Reference order ids. No preamble."
        )}],
    )
    return "".join(b.text for b in r.content if b.type == "text")   # the brief text

**This cell:** runs the synthesiser. Live, it prints the real brief built from the
fan-out results; offline, it prints an illustrative example so you can see the shape of the
output the fan-in produces.

In [ ]:
# ===== run the synthesis =====
if RUN_LIVE:                                       # live: write the real brief
    print(synthesize(summaries, rollup))           #   one synthesis call over the fan-out results
else:                                              # offline: describe what it would do
    print("[offline] synthesis needs a key. It would turn the rollup above into a brief like:")
    print("  5 tickets today: 2 refunds, 2 shipping, 1 address. A3 and A4 are high priority.")

**This cell:** the **caching report**. It sums the tokens written to cache versus
read from cache across the fan-out. A read costs a small fraction of a normal input token,
so the read total is the part you stopped paying full price for by caching the policy.

In [ ]:
# ===== measure the caching payoff =====
if RUN_LIVE and usages:                           # only meaningful on a live run
    created = sum(u["created"] for u in usages)   #   total tokens WRITTEN to cache (paid once)
    read = sum(u["read"] for u in usages)         #   total tokens READ from cache (cheap reuse)
    print("cache created (written once):", created)   # expect the policy size on the first call
    print("cache read   (reused cheaply):", read)     # expect roughly policy size times (N - 1)
    print("cache reads cost about 0.1x a normal input token, so this is the saving.")
else:                                              # offline explanation
    print("[offline] on a live run you would see the policy WRITTEN on call 1")
    print("          and READ on the other calls; reads bill at about 0.1x input price.")

| anti-pattern | what to do instead |
|---|---|
| paste every ticket into one giant prompt | fan out: one focused call per ticket, so attention stays sharp |
| stitch prose answers together by hand | pass structured JSON between stages, then aggregate in code |
| resend the full policy on every call | cache it once with `cache_control`; the rest read it cheaply |
| fan out five cold calls and hope they cache | warm the cache with one call first, then parallelise the rest |

**Lesson:** split independent work so each call sees only what it needs (relevance
beats volume), pass **JSON** between stages so fan-in is deterministic, and **cache** any
context that repeats. Fan-out gives you speed and focus; fan-in gives you one coherent
result; caching gives you the same context at a fraction of the cost.

---

## Recap - fan-out, fan-in, and caching

| Stage | What ShopDesk does | Course topic |
|---|---|---|
| Fan-out | one parallel call per ticket, each with only that ticket plus the policy | decomposing into parallel sub-prompts |
| Intermediate JSON | small structured summaries hand off between stages | JSON for intermediate results |
| Fan-in (aggregate) | group the summaries in pure Python | aggregating parallel results |
| Fan-in (synthesise) | one call turns the rollup into a brief | synthesising into one result |
| Prompt caching | the shared policy is written once, read cheaply after | keeping sessions efficient |

One principle ties them together: **parallelise what is independent, merge what belongs
together, and never pay twice for the same context.** To run live, paste a real key into
**Setup 2/3** and re-run from the top. Then try it: add two more tickets and watch the
cache-read total grow while the written total stays flat.